# Projection-based WF-in-DFT embedding

A clean end-to-end run of $\mu$-shift projection embedding: a cheap DFT calculation on the whole molecule, then a correlated wavefunction on a fragment of it, with the environment kept at the DFT level.

The physics follows Manby *et al.*,
[JCTC **8**, 2564 (2012)](https://doi.org/10.1021/ct300544e), and the energy expression is
Eq. 8 of Goodpaster *et al.*,
[JCP **140**, 18A507 (2014)](https://doi.org/10.1063/1.4864040).

**Roadmap**

1. Partition the global DFT energy into fragment, environment and cross terms.
2. Choose which occupied orbitals belong to the fragment (`code/active_space.py`).
3. Build the embedding potential $v_\text{emb}$ and the environment projector $\hat P_B$.
4. Validate the machinery with DFT-in-DFT, which must reproduce the global DFT energy exactly.
5. Swap the fragment solver for HF, CISD, CCSD and CASCI.
6. Check the exact limit: with the whole molecule as the fragment, every WF-in-DFT number
   must collapse onto the corresponding whole-molecule wavefunction energy.

**Three things that silently give wrong answers**, all of which bit notebook 23 and each of
which is called out at the point it matters below:

- the projector must span the *occupied* environment orbitals only;
- the embedding correction contracts $v_\text{emb}$ with the *DFT* fragment density, not with
  the solver's density;
- an active space must be chosen from the *post-embedding* orbitals, never by reusing
  orbital indices from the global calculation.

Section 6 also looks at what deleting the level-shifted orbitals does and does not buy, which
turns out to be a cost and resource question rather than a correctness one.

In [1]:
import numpy as np
from pyscf import gto, scf, lo, ci, cc, mcscf, dft, fci

## 1. Splitting the DFT energy in two

Run KS-DFT on the whole molecule and split its occupied orbitals into two disjoint sets, a
fragment $A$ and an environment $B$. Because the sets are disjoint and the orbitals are
orthonormal, the total density splits exactly:

$$\gamma = \gamma_A + \gamma_B .$$

The energy does not split exactly, because it is not linear in the density. Define the
leftover as the non-additive energy:

$$E_\text{nad}[\gamma_A,\gamma_B] \;\equiv\; E_\text{DFT}[\gamma_A+\gamma_B] - E_\text{DFT}[\gamma_A] - E_\text{DFT}[\gamma_B],$$

so that

$$E_\text{DFT}[\gamma] = E_\text{DFT}[\gamma_A] + E_\text{DFT}[\gamma_B] + E_\text{nad}[\gamma_A,\gamma_B].$$

Nothing is approximated yet; $E_\text{nad}$ is defined by this equation. It collects the
inter-subsystem Coulomb repulsion and the non-additive exchange-correlation energy.

The reason to build the partition from *orbitals* rather than from densities alone is that
there is **no non-additive kinetic energy term** here. Both subsystems are described by
orthonormal orbital sets drawn from one Slater determinant, so the kinetic energy is additive
by construction. Approximating the non-additive kinetic energy is the main error source in
pure density-based (frozen-density) embedding, and projection-based embedding avoids it
entirely. That is what makes DFT-in-DFT exact here, which we will verify numerically.

In the code below, energies are electronic only (`energy_elec`, which excludes nuclear
repulsion), and each subsystem density is evaluated in the field of *all* the nuclei, so
$E_\text{nuc}$ enters the total exactly once:

- `E_act_dft` $= E_\text{DFT}[\gamma_A]$
- `E_env` $= E_\text{DFT}[\gamma_B]$
- `E_cross` $= E_\text{nad}[\gamma_A,\gamma_B]$

### The system

Methanol, with the hydroxyl group as the fragment. `6-31G` and `b3lyp` keep every step in this
notebook to a second or two, so the whole thing can be re-run while changing the partition.

In [2]:
geometry = [
    ("O", (-0.6582, -0.0067,  0.1730)),
    ("H", (-1.1326, -0.0311, -0.6482)),
    ("C", ( 0.7031,  0.0083, -0.1305)),
    ("H", ( 0.9877,  0.8943, -0.7114)),
    ("H", ( 1.0155, -0.8918, -0.6742)),
    ("H", ( 1.2001,  0.0363,  0.8431)),
]


basis_set = "6-31G"
xc        = "b3lyp"
charge    = 0
spin      = 2         # 2S
max_memory = 10_000    # Mb

# the fragment: the hydroxyl O and H (0-based atom indices)
active_atm_idx = [0, 1]
n_occ_active   = 3     # occupied orbitals given to the fragment -> its electron count
n_vir_active   = 3     # extra virtuals in the fragment block -> sets the CAS size
mu_val         = 1e6   # level shift

mol = gto.Mole(
    atom=geometry,
    basis=basis_set,
    charge=charge,
    spin=spin,
    max_memory=max_memory,
).build()

mol.nao, mol.nelec

(26, (10, 8))

In [3]:
global_scf = dft.ROKS(mol, xc=xc)
global_scf.nelec

(10, 8)

In [4]:
## the cheap calculation on the whole molecule: everything downstream is built from this
global_scf = scf.ROKS(mol, xc=xc)
global_scf.kernel()

## whole-molecule wavefunction references, for context later on
mf_hf = scf.RHF(mol).run()
ci_full = ci.CISD(mf_hf).run()
cc_full = cc.CCSD(mf_hf).run()

print(f"\nwhole molecule: B3LYP {global_scf.e_tot:.8f}   RHF {mf_hf.e_tot:.8f}   "
      f"CISD {ci_full.e_tot:.8f}   CCSD {cc_full.e_tot:.8f}")

converged SCF energy = -115.403719541911
converged SCF energy = -114.720928305621

WARN: RCISD method does not support ROHF method. ROHF object is converted to UHF object and UCISD method is called.

E(UCISD) = -114.922449556955  E_corr = -0.2015212513337987

WARN: RCCSD method does not support ROHF method. ROHF object is converted to UHF object and UCCSD method is called.

E(UCCSD) = -114.9379881253072  E_corr = -0.2170598196860483

whole molecule: B3LYP -115.40371954   RHF -114.72092831   CISD -114.92244956   CCSD -114.93798813


## 2. Which orbitals belong to the fragment

`active_space.py` scores every MO by how much of it sits on the target atoms, using a
**Löwdin population**. Orthogonalise the AO basis with $S^{1/2}$, so that squared coefficients
genuinely partition an orbital, and sum them over the target atoms' AOs:

$$w_i \;=\; \sum_{\mu \in A} \left[ (S^{1/2} C)_{\mu i} \right]^2 \;\in\; [0,1].$$

Because the basis is orthogonalised first, $\sum_\mu$ over all atoms gives exactly 1 for every
orbital, so $w_i$ reads directly as "the fraction of orbital $i$ living on the fragment". A raw
Mulliken sum over the same AOs is not bounded this way and can even go negative. Core $1s$ AOs
are dropped for elements beyond helium so that tight cores cannot dominate the score; H keeps
its $1s$, which is its valence shell.

Occupied and virtual orbitals are ranked separately, and no MO-index window is used: a strongly
fragment-localised orbital high in the virtual space is usually a compact $\sigma^*$, exactly
what bond breaking needs, and `orbital_spread` (the RMS extent $\sqrt{\langle r^2\rangle - \langle r\rangle^2}$)
is the honest way to reject genuinely diffuse Rydberg-like orbitals instead.

**Only the occupied partition affects the embedding.** The fragment occupied orbitals fix the
subsystem electron count, and the environment occupied orbitals build the projector. The
selected *virtuals* never enter the embedded SCF, because after embedding the subsystem has its
own virtual space; they only fix how big a CAS we hand to a post-embedding solver.

### Why localise first

Canonical KS orbitals are delocalised, so no single one of them is "the O–H bond": the split is
fuzzy and the fragment ends up sharing orbitals with the environment. A unitary rotation
*within* the occupied block leaves $\gamma$, and therefore the global DFT energy, completely
unchanged, but it makes the orbitals atom-centred and the partition clean. That is free
accuracy, so do it. Below, Pipek-Mezey is applied separately to the occupied and virtual
blocks, which keeps `mo_occ` meaningful column by column.

In [5]:
## localise inside the occupied and virtual blocks separately, so that mo_occ still
## describes column i, and the total density is untouched

## works with open shell!
occ_double_mask = global_scf.mo_occ > 1
occ_single_mask = global_scf.mo_occ == 1
vir_mask = global_scf.mo_occ == 1

C_loc = global_scf.mo_coeff.copy()
C_loc[:,  occ_double_mask] = lo.PipekMezey(mol, global_scf.mo_coeff[:,  occ_double_mask]).kernel()
C_loc[:,  occ_single_mask] = lo.PipekMezey(mol, global_scf.mo_coeff[:,  occ_single_mask]).kernel()
C_loc[:,  vir_mask] = lo.PipekMezey(mol, global_scf.mo_coeff[:,  vir_mask]).kernel()


## a unitary rotation within the occupied block must leave the density alone
assert np.allclose(
    global_scf.make_rdm1(mo_coeff=C_loc, mo_occ=global_scf.mo_occ),
    global_scf.make_rdm1(mo_coeff=global_scf.mo_coeff, mo_occ=global_scf.mo_occ),
    atol=1e-9,
), "localisation changed the density"


### What the partition object holds

`select_active_space` returns the orbitals reordered as

$$C_\text{active} = [\;\underbrace{\text{env occ}}_{n_\text{core}}\;|\;\underbrace{\text{frag occ}\;|\;\text{frag vir}}_{n_\text{cas}}\;|\;\text{remaining vir}\;]$$

so that `act_cols` and `env_cols` index *this* ordering. Two attributes matter most:

- `env_occ_cols` — the occupied environment orbitals, which build the projector;
- `n_env_mo` — how many orbitals get level-shifted, and so how many a correlated solver
  must discard later.

`orig_active_idxs` records the fragment orbitals in the original numbering. It is for
diagnostics only: those indices refer to the *global* orbital set and are meaningless as a
selector once the embedded SCF has produced its own orbitals.

`space.densities()` builds $\gamma$, $\gamma_A$ and $\gamma_B$ from the same reordered
coefficients, and asserts that they are additive and hold the right electron counts.

In [6]:
from nbed.act_env_space import lowdin_populations, orbital_spread

In [7]:
C = C_loc
# C =  global_scf.mo_coeff.copy()


per_atom, population = lowdin_populations(mol, C, active_atm_idx, drop_core_1s=True)
spread = orbital_spread(mol, C)

eligible = np.ones_like(population, dtype=bool)

# max_spread: Reject orbitals more diffuse than this, in BOHR.
max_spread = 2
if max_spread is not None:
    eligible &= spread <= max_spread


occ_pool = np.where((global_scf.mo_occ > 0) & eligible)[0]
vir_pool = np.where((global_scf.mo_occ == 0) & eligible)[0]

# if len(occ_pool) < n_occ_active or len(vir_pool) < n_vir_active:
#     raise ValueError(
#         f"asked for {n_occ_active} occupied and {n_vir_active} virtual "
#         f"orbitals but only {len(occ_pool)} and {len(vir_pool)} are "
#         "eligible; relax max_spread or shrink the fragment"
#     )

## get most important occupied and virtual orbitals
pick_occ = np.sort(occ_pool[np.argsort(-population[occ_pool])[:n_occ_active]])
pick_vir = np.sort(vir_pool[np.argsort(-population[vir_pool])[:n_vir_active]])

In [8]:
acitve_indices  = np.concatenate([pick_occ, pick_vir])
enviro_indices = np.setdiff1d(np.arange(global_scf.mol.nao), acitve_indices)


In [9]:
from nbed.emb_scf import EmbedSCF

In [10]:
len(acitve_indices) + len(enviro_indices)

26

In [11]:
global_scf.mo_occ.shape

(26,)

In [12]:
Sao = global_scf.get_ovlp()
emb_obj = EmbedSCF(global_scf,
                 acitve_indices, enviro_indices, 
                  C, 
                  global_scf.mo_occ,
                  Sao,
                   10_000, 
                   mu_val=1e9)

In [13]:
efull, emb_rks, corr, env_cols, env_plus_corrections = emb_obj.build_emb_dft("B3LYP", proj_type="mu")
efull

Overwritten attributes  get_hcore  of <class 'pyscf.dft.roks.ROKS'>


converged SCF energy = -38.8619953459077


np.float64(-115.40371950063724)

In [14]:
emb_obj.check_embedding(emb_rks.mo_coeff,
                         emb_rks.mo_occ,
                          emb_rks.mo_energy,
                           Sao, 
                           "mu")

--- mu ---
  environment landed in columns : [19 20 21 22 23 24 25]
  eps(environment)              : [9.99999999e+08 9.99999999e+08 9.99999999e+08 1.00000000e+09
 1.00000000e+09 1.00000000e+09 1.00000000e+09]
  occupied columns              : [0 1 2]
  eps(occupied)                 : [-19.2696 -10.2562  -0.5551]
  max |<env occ| S |emb occ>|   : 7.86e-11   <- must be ~0
  aufbau margin                 : +999999999.4343 Ha  <- must be > 0


array([19, 20, 21, 22, 23, 24, 25])

In [15]:
## need this to be all zero!
## shows env orbitals are orthogonal to optimized act orbitals!
non_env_idxs = np.setdiff1d(np.arange(global_scf.mol.nao), env_cols)
np.around(emb_rks.mo_coeff[:, non_env_idxs].conj().T @ Sao @ emb_obj.C_full_reidx[:, emb_obj.env_idx_occ], 
    8)

array([[-0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [-0.,  0., -0., -0., -0., -0.,  0.],
       [ 0., -0., -0.,  0.,  0.,  0.,  0.],
       [-0.,  0.,  0., -0., -0.,  0.,  0.],
       [-0., -0.,  0., -0.,  0.,  0., -0.],
       [ 0., -0., -0.,  0.,  0., -0.,  0.],
       [ 0.,  0., -0., -0., -0.,  0.,  0.],
       [-0.,  0.,  0.,  0., -0.,  0.,  0.],
       [ 0., -0., -0., -0., -0., -0., -0.],
       [-0., -0., -0., -0.,  0., -0., -0.],
       [-0., -0.,  0., -0.,  0., -0.,  0.],
       [ 0., -0., -0., -0., -0.,  0.,  0.],
       [-0.,  0.,  0.,  0., -0., -0.,  0.],
       [-0., -0., -0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -0.,  0.,  0., -0.,  0.],
       [-0.,  0., -0., -0., -0.,  0.,  0.],
       [-0., -0.,  0.,  0.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  0., -0.,  0.,  0.],
       [-0.,  0., -0., -0., -0., -0.,  0.]])

In [43]:
efull2, emb_rks2, corr2, env_cols2, env_plus_corrections2 = emb_obj.build_emb_dft("B3LYP", proj_type="huz", 
                                            huz_level_shift=0)
efull2


converged SCF energy = -38.8619953459077


np.float64(-115.40371950063724)

In [17]:
P_huz = emb_obj.get_huz_projector()
overlap_full_proj = emb_obj.C_full_reidx.T @ Sao @  P_huz @ emb_obj.C_full_reidx
## to 10 sig figs!
non_ven_occ = np.setdiff1d(np.arange(global_scf.mol.nao), emb_obj.env_idx_occ)
print("act cols:    ", np.around(np.diag(overlap_full_proj), 10)[non_ven_occ])
print("env cols:    ", np.around(np.diag(overlap_full_proj), 10)[emb_obj.env_idx_occ])

act cols:     [ 0.  0.  0.  0. -0. -0.  0.  0. -0. -0.  0. -0. -0. -0. -0. -0. -0.  0.
 -0.]
env cols:     [1. 1. 1. 1. 1. 1. 1.]


In [18]:
emb_obj.check_embedding(emb_rks2.mo_coeff,
                         emb_rks2.mo_occ,
                          emb_rks2.mo_energy,
                           Sao, 
                           "huz")

--- huz ---
  environment landed in columns : [ 3  8  9 10 14 15 24]
  eps(environment)              : [-0.0363  0.3239  0.4107  0.49    0.5563  0.6925  1.1208]
  occupied columns              : [0 1 2]
  eps(occupied)                 : [-19.2696 -10.2562  -0.5551]
  max |<env occ| S |emb occ>|   : 8.99e-16   <- must be ~0
  aufbau margin                 : +0.5188 Ha  <- must be > 0


array([ 3,  8,  9, 10, 14, 15, 24])

In [19]:
# One carry-over for the WF step: with Huzinaga the
#  environment orbitals are interleaved with the active virtuals
#  (columns [3, 8, 9, 10, 14, 15, 24] here), 
# so when you drop them for CCSD/CASCI select them 
# by projection weight the way check_embedding does, not by column position. (i.e. USE: env_cols2)


## need this to be all zero!
## shows env orbitals are orthogonal to optimized act orbitals!
non_env_idxs = np.setdiff1d(np.arange(global_scf.mol.nao), env_cols2)
np.around(emb_rks2.mo_coeff[:, non_env_idxs].conj().T @ Sao @ emb_obj.C_full_reidx[:, emb_obj.env_idx_occ], 
    12)

array([[ 0., -0.,  0.,  0.,  0., -0.,  0.],
       [-0., -0.,  0., -0., -0., -0.,  0.],
       [-0., -0., -0., -0., -0., -0.,  0.],
       [ 0., -0., -0.,  0., -0.,  0.,  0.],
       [ 0., -0.,  0.,  0., -0., -0., -0.],
       [ 0., -0., -0.,  0., -0., -0., -0.],
       [ 0., -0., -0.,  0.,  0., -0.,  0.],
       [-0., -0.,  0., -0.,  0.,  0., -0.],
       [-0., -0., -0.,  0.,  0., -0., -0.],
       [ 0.,  0.,  0., -0., -0., -0., -0.],
       [-0., -0.,  0., -0., -0.,  0.,  0.],
       [-0., -0.,  0., -0., -0., -0., -0.],
       [-0., -0., -0., -0.,  0., -0., -0.],
       [ 0., -0.,  0., -0., -0.,  0., -0.],
       [-0., -0.,  0., -0., -0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -0.,  0.],
       [-0.,  0., -0., -0., -0.,  0.,  0.],
       [-0.,  0., -0.,  0., -0., -0.,  0.],
       [-0., -0., -0., -0.,  0.,  0., -0.]])

In [ ]:
efull3, emb_rhf, corr3, env_cols3, env_plus_corrections3 = emb_obj.build_emb_hf(proj_type="huz")


non_env_idxs = np.setdiff1d(np.arange(global_scf.mol.nao), env_cols3)
np.around(emb_rhf.mo_coeff[:, non_env_idxs].conj().T @ Sao @ emb_obj.C_full_reidx[:, emb_obj.env_idx_occ], 
          12)

converged SCF energy = -38.7290919691101


Overwritten attributes  get_hcore  of <class 'pyscf.scf.rohf.ROHF'>


array([[-4.30e-11,  4.40e-11,  1.10e-11,  2.00e-12,  2.00e-12, -0.00e+00,
         1.90e-11],
       [-2.90e-11,  2.00e-12, -1.00e-12, -3.70e-11, -3.70e-11, -1.00e-12,
         2.00e-12],
       [ 8.00e-11, -2.30e-11, -2.00e-11,  7.60e-11,  8.40e-11, -4.00e-11,
         1.00e-12],
       [ 7.00e-12, -2.50e-11, -4.00e-12, -4.40e-11, -3.40e-11, -1.70e-11,
        -1.00e-12],
       [ 7.00e-12,  5.00e-12, -8.00e-12, -3.80e-11,  4.60e-11, -1.70e-11,
        -1.00e-11],
       [-3.00e-12,  7.00e-12,  5.00e-12, -1.00e-11, -8.00e-12, -2.40e-11,
         0.00e+00],
       [-4.40e-11, -3.60e-11, -2.00e-11,  6.00e-12,  1.30e-11, -4.00e-12,
         3.50e-11],
       [-5.00e-12, -4.00e-12,  1.20e-11, -4.80e-11,  1.70e-11,  1.20e-11,
         6.00e-12],
       [ 1.00e-12, -8.00e-12,  2.00e-11,  1.00e-11,  4.60e-11, -7.00e-12,
         1.10e-11],
       [-2.30e-11, -4.00e-12,  1.20e-11, -8.00e-12,  1.30e-11,  3.00e-12,
        -4.80e-11],
       [ 1.60e-11, -4.20e-11, -4.00e-11,  2.20e-11, -2.30e-1

In [45]:
emb_rhf.mol.nao, emb_rhf.mol.nelec

(26, (3, 3))

In [46]:
ncas    = 6
nelecas = (2,2)
mycas = mcscf.CASCI(emb_rhf, ncas, nelecas)
mycas.kernel()
mycas.e_tot + env_plus_corrections3


CASCI E = -38.7322386508985  E(CI) = -30.5931882356069  S^2 = 0.0000000


np.float64(-115.27396280562812)

In [47]:
C_act = emb_rhf.mo_coeff


ncore = (emb_rhf.mol.nelectron - np.sum(nelecas) )// 2 ## /2 for double occ!
mo_cas_idxs = np.arange(ncore, ncore + ncas)

ecore, h1e, eri = emb_obj.get_mo_integrals(
            emb_rhf, 
            C_act,
            ncas,
            nelecas,
            mo_cas_idxs= mo_cas_idxs
            )



In [48]:
from nbed.hamiltonian import build_spin_integrals, build_molecular_H, build_number_operator

h1e_spin, eri_spin = build_spin_integrals(h1e, eri, ncas)
e_shift = env_plus_corrections3 + ecore

nqubits = 2*ncas

Hq = build_molecular_H(e_shift, h1e_spin, eri_spin)
Na, Nb = build_number_operator(nqubits, type="qubit_jw")

In [ ]:
from scipy.sparse.linalg import eigsh
from openfermion import get_sparse_operator
if nqubits<13:
    H_sym = Hq + (Na - nelecas[0])**2 + (Nb - nelecas[1])**2
    Hq_mat = get_sparse_operator(H_sym).real
    eigvals, eigvecs = eigsh(Hq_mat, k=3, which="SA")


In [ ]:
eigvals[0] - (mycas.e_tot + env_plus_corrections3)

np.float64(-4.405364961712621e-13)